In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib
import re

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 15
df = pd.read_csv("meta_unique.tsv", sep='\t')

In [ ]:
row_mapping = {
    'plasma_tubes_short_name': 'Blood collection tube',
    'centrifugation_step_1': 'Centrifugation, step 1',
    'centrifugation_step_2': 'Centrifugation, step 2',
    'tissue': 'Biomaterial',
    'rna_extraction_kit_short_name': 'RNA extraction kit',
    'dnase': 'DNAse treatment',
    'library_prep_kit_short_name': 'Library prep kit',
    'library_selection': 'Library selection',
    'cdna_library_type': 'cDNA library type',
    'read_length': 'Read length',
    'broad_protocol_category': 'Broad protocol category (BPC)',
    'plasma_volume':'Plasma Volume (mL)',
    'UMI': 'UMI',
    'biomaterial_provider':'Bio Material Provider',
    'Batch_Info_1': 'Batch 1',
    'Batch_Info_2': 'Batch 2',
    'Center Name':'Center name',
}

plot_df = df[list(row_mapping.keys()) + ['author']].rename(columns=row_mapping).set_index('author').T
plot_df = plot_df.fillna("Blank").astype(str)

NULL_ORDER = ['None', 'Unknown', 'Unspecified', 'Missing', 'Blank']

def extract_num(s):
    nums = re.findall(r'\d+', str(s))
    return int(nums[-1]) if nums else -1

unique_plot_df = plot_df.copy()
for row in plot_df.index:
    unique_plot_df.loc[row] = row + "::" + plot_df.loc[row]

all_unique_keys = sorted(list(set(unique_plot_df.values.ravel())))
def custom_sort_key(key):
    row, val = key.split("::")
    if val in NULL_ORDER:
        return (1, NULL_ORDER.index(val), val)
    digits = "".join([c for c in val if c.isdigit()])
    if digits:
        return (0, 0, extract_num(val))
    return (0, 1, val.lower())

all_unique_keys = sorted(all_unique_keys, key=custom_sort_key)
master_palette = {}

categorical_palettes = ['Set2', 'Set1', 'Pastel1', 'Set3', 'Spectral', 'tab10', 'tab20']
pal_idx = 0

for row in plot_df.index:
    vals = sorted([v for v in plot_df.loc[row].unique()], key=lambda x: custom_sort_key(row + "::" + x))
    
    null_colors = {'Unknown': "#909090", 'Unspecified': "#878787", 'None': "#FFFFFF", 'Missing': '#A6A6A6', 'Blank': '#FFFFFF'}
    if row in ['Centrifugation, step 1', 'Centrifugation, step 2', 'Read length', 'Plasma Volume (mL)']:
        cmap_name = {'Centrifugation, step 1': 'Oranges', 'Centrifugation, step 2': 'Reds', 
                     'Read length': 'Greens', 'Plasma Volume (mL)': 'Purples'}[row]
        nums = [v for v in vals if v not in NULL_ORDER]
        cmap = plt.get_cmap(cmap_name)
        for i, v in enumerate(nums):
            color = cmap(0.35 + (0.55 * (i / (len(nums)-1)))) if len(nums) > 1 else cmap(0.5)
            master_palette[row + "::" + v] = color
            
    # (C) 특수 변수 (BPC)
    elif row == 'Broad protocol category (BPC)':
        bpc_map = {'Whole RNA-Seq (random pr.) (WRR)': '#E41A1C', 'Whole RNA-Seq (oligo-dT pr.) (WRO)': '#377EB8',
                   'Exome-based (EB)': '#4DAF4A', 'Custom': '#984EA3'}
        for v in vals:
            if v in bpc_map: master_palette[row + "::" + v] = bpc_map[v]

    # (D) 일반 범주형 변수 (각기 다른 팔레트 적용)
    else:
        current_pal = sns.color_palette(categorical_palettes[pal_idx % len(categorical_palettes)], len(vals))
        pal_idx += 1
        for i, v in enumerate(vals):
            if v not in master_palette: # 이미 지정된 특수색 제외
                master_palette[row + "::" + v] = current_pal[i]

    # (E) 공통 예외 처리: Streck은 항상 DarkRed, 결측치는 회색
    for v in vals:
        if v == 'Streck': master_palette[row + "::" + v] = '#8B0000'
        if v in null_colors: master_palette[row + "::" + v] = null_colors[v]

# 데이터 수치화
cat_to_int = {val: i for i, val in enumerate(all_unique_keys)}
numeric_df = unique_plot_df.applymap(lambda x: cat_to_int.get(x))
custom_cmap = ListedColormap([master_palette[key] for key in all_unique_keys])

# --- 3. 플로팅 (사이즈 및 레이아웃 옵션 유지) ---
fig, ax = plt.subplots(figsize=(35, 15))
sns.heatmap(numeric_df, cmap=custom_cmap, cbar=False, linewidths=0.5, linecolor='white', ax=ax)
ax.set_xticklabels([])
ax.set_xlabel('')

# Author 브래킷 로직 유지
authors = plot_df.columns.tolist()
unique_authors = []
last_author, start_idx = None, 0
for i, author in enumerate(authors):
    if author != last_author:
        if last_author is not None:
            unique_authors.append({'name': last_author, 'start': start_idx, 'end': i - 1})
        start_idx, last_author = i, author
unique_authors.append({'name': last_author, 'start': start_idx, 'end': len(authors) - 1})

for group in unique_authors:
    x_s, x_e = group['start'] + 0.1, group['end'] + 0.9
    y_p = numeric_df.shape[0] + 0.5
    ax.plot([x_s, x_s, x_e, x_e], [y_p, y_p+0.3, y_p+0.3, y_p], color='black', lw=1.5, clip_on=False)
    ax.text((x_s+x_e)/2, y_p + 0.6, group['name'], ha='center', va='top', fontsize=15, rotation=90)

# 범례 설정 유지
legend_ax = fig.add_axes([0.68, 0.05, 0.30, 0.9]) 
legend_ax.axis('off')
legend_ax.set_xlim(0, 1.2) 

rows = list(plot_df.index)
x_offsets = [0.0, 0.45, 0.90] 
n_cols = 3 ; r_per_c = int(np.ceil(len(rows) / n_cols))

for c_idx in range(n_cols):
    cx, cy = x_offsets[c_idx], 0.98
    start_r = c_idx * r_per_c
    for r_idx in range(start_r, min(start_r + r_per_c, len(rows))):
        row_name = rows[r_idx]
        legend_ax.text(cx, cy, row_name, fontweight='bold', fontsize=16)
        cy -= 0.018
        # 해당 행의 원래 값들로 범례 표시
        row_vals = sorted(list(plot_df.loc[row_name].unique()), key=lambda x: custom_sort_key(row_name + "::" + x))
        for v in row_vals:
            rect = plt.Rectangle((cx, cy - 0.012), 0.025, 0.018, color=master_palette[row_name + "::" + v], ec='#444444', lw=0.6)
            legend_ax.add_patch(rect)
            legend_ax.text(cx + 0.035, cy - 0.005, v, fontsize=14, va='center')
            cy -= 0.018
        cy -= 0.02 

plt.subplots_adjust(left=0.04, right=0.66, bottom=0.2)
plt.savefig("Unique_Preanalytical_Combinations.png", dpi=300, bbox_inches='tight', pad_inches=0.5)